# Transformer Internals

**Topics Covered:**
- Attention
- Multi-Head Attention
- Positional Encoding

These are the core ideas behind models like:
- OpenAI's GPT models
- Google's BERT
- Most modern LLMs

## 1. Why Transformers Were Created

Before transformers, models used:
- **RNNs**
- **LSTMs**

**Problems with them:**
- Process words one by one
- Slow training
- Hard to remember long context

**Example:**

> *"The animal didn't cross the street because it was too tired."*

To understand **"it"**, the model must remember **animal** from earlier. RNNs struggled with long-range memory.

---

### Transformer Solution

Transformers introduced the **Attention Mechanism**.

Instead of reading sequentially → **every word can look at every other word directly.**

This changed everything.

## 2. What is Attention?

### Human Analogy

> *"The cat sat on the mat because it was soft."*

When understanding **"it"**, your brain asks:
- What does "it" refer to? Cat? Mat?
- You pay more attention to **mat**.

That is exactly what attention does.

---

### Definition

> **Attention** allows each word to **dynamically focus on the most relevant words** in the sentence.

---

### Core Idea

For every token:
1. Compare it with all other tokens
2. Decide relevance scores
3. Use a weighted combination of important tokens

---

### Example

> *"I went to the bank to deposit money."*

When processing **bank**, attention focuses on:
- **deposit**
- **money**

So the model understands: **bank = financial institution**

## 3. How Attention Works Internally

This is where **Q, K, V** come in.

---

### Every Token Creates 3 Vectors

For each word embedding, the Transformer produces:

| Vector | Name | Intuition |
|--------|------|-----------|
| **Q** | Query | *"What am I looking for?"* |
| **K** | Key | *"What do I contain?"* |
| **V** | Value | *"What information should I provide?"* |

These are generated using **learned weight matrices**.

---

### Real-World Analogy: Library Search

Your current word asks:
- **Query:** *"I need context about financial meaning"*

Each other word provides:
- **Key:** *"I relate to finance"*
- **Value:** Actual contextual information

---

### Attention Calculation

**Step 1 — Compare Query with all Keys (dot product):**

$$\text{Score} = Q \cdot K$$

Higher score = more relevance.

**Step 2 — Normalize scores with Softmax:**

Turns scores into probabilities.

| Word | Score | Softmax |
|------|-------|---------|
| deposit | 8 | 0.65 |
| money | 6 | 0.30 |
| went | 1 | 0.05 |

**Step 3 — Weighted Sum of Values:**

$$\text{Output} = \sum \text{AttentionWeight}_i \times \text{Value}_i$$

---

### Final Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

---

### Why divide by $\sqrt{d_k}$?

Without scaling:
- Dot products become too large
- Softmax becomes unstable
- Training suffers

**Scaling ensures numerical stability.**

In [ ]:
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores)
    output = weights @ V
    return output, weights

# Example: processing "bank" with context words [deposit, money, went]
# Each word is a simple 4-dim embedding (for illustration)
np.random.seed(42)
Q = np.random.randn(1, 4)       # query for "bank"
K = np.random.randn(3, 4)       # keys for [deposit, money, went]
V = np.random.randn(3, 4)       # values for [deposit, money, went]

output, weights = scaled_dot_product_attention(Q, K, V)

words = ["deposit", "money", "went"]
print("Attention weights for 'bank':")
for word, w in zip(words, weights[0]):
    print(f"  {word:10s}: {w:.4f}")

print(f"\nContext vector (output): {output[0]}")

## 4. Self-Attention

### Meaning

When attention happens **within the same sentence/input** — each token attends to other tokens in the same sequence — it is called **Self-Attention**.

---

### Example

> *"The dog chased the ball because it was fast."*

When processing **"it"**, self-attention helps determine:
- **it = dog?**
- **it = ball?**

The model attends to all tokens in the same sentence to resolve the reference.

## 5. Multi-Head Attention

### Problem with Single Attention Head

One attention mechanism may focus only on one type of relationship at a time:
- Syntax **OR**
- Semantics **OR**
- Position

But not all well simultaneously.

---

### Solution: Multiple Attention Heads

> Use **Multiple Attention Heads** — each head learns different relationships.

---

### Example

> *"The boy who wore red shoes kicked the ball."*

| Head | Focus | Relationship |
|------|-------|--------------|
| Head 1 | Grammar | boy ↔ kicked |
| Head 2 | Attributes | boy ↔ red shoes |
| Head 3 | Object relation | kicked ↔ ball |

---

### Architecture

Instead of one Q/K/V projection, create **multiple sets**:

```
Head 1 → Q₁, K₁, V₁
Head 2 → Q₂, K₂, V₂
Head 3 → Q₃, K₃, V₃
```

Each head learns different patterns independently.

---

### Process

1. Run attention independently per head
2. Concatenate outputs
3. Project back to original dimension

---

### Formula

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \, W^O$$

where each head is:

$$\text{head}_i = \text{Attention}(Q W_i^Q,\ K W_i^K,\ V W_i^V)$$

---

### Why Multi-Head Helps

Language has many **simultaneous relationships**:
- Grammar
- Semantics
- Coreference
- Position
- Long-range dependencies

Multiple heads capture them **in parallel**.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)
    weights = softmax(scores)
    return weights @ V, weights

class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads

        np.random.seed(0)
        self.W_Q = [np.random.randn(d_model, self.d_k) for _ in range(num_heads)]
        self.W_K = [np.random.randn(d_model, self.d_k) for _ in range(num_heads)]
        self.W_V = [np.random.randn(d_model, self.d_k) for _ in range(num_heads)]
        self.W_O = np.random.randn(d_model, d_model)

    def forward(self, X):
        # X: (seq_len, d_model)
        heads = []
        for i in range(self.num_heads):
            Q = (X @ self.W_Q[i])[np.newaxis]  # (1, seq_len, d_k)
            K = (X @ self.W_K[i])[np.newaxis]
            V = (X @ self.W_V[i])[np.newaxis]
            head_out, _ = attention(Q, K, V)
            heads.append(head_out[0])  # (seq_len, d_k)

        concat = np.concatenate(heads, axis=-1)  # (seq_len, d_model)
        return concat @ self.W_O


# Demo
d_model, num_heads, seq_len = 8, 2, 5
X = np.random.randn(seq_len, d_model)

mha = MultiHeadAttention(d_model, num_heads)
output = mha.forward(X)

print(f"Input  shape: {X.shape}       → (seq_len={seq_len}, d_model={d_model})")
print(f"Output shape: {output.shape}  → same shape preserved after multi-head attention")

## 6. Positional Encoding

### Problem

Transformer processes **all tokens simultaneously**.

So unlike RNNs, it has **no natural understanding of word order**.

Sentence meanings change with order:
- *"Dog bites man"*
- *"Man bites dog"*

Without position info → Transformer sees the same **bag of words**.

---

### Solution: Add Positional Encoding

Inject position information directly into the embeddings:

$$\text{FinalEmbedding} = \text{TokenEmbedding} + \text{PositionalEncoding}$$

---

### Fixed Sinusoidal Encoding (Original Paper)

Uses **sine/cosine waves** at different frequencies:

$$PE_{(pos,\ 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$

$$PE_{(pos,\ 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

Where:
- $pos$ = position of the token in the sequence
- $i$ = dimension index
- $d$ = embedding dimension

---

### Why Sine/Cosine?

- Allows the model to learn **relative distances** between tokens
- Captures **pattern periodicity**
- Can **extrapolate to longer sequences** than seen during training

---

### Modern Alternatives

Many newer LLMs use learned or relative encodings:

| Method | Used In |
|--------|---------|
| Learned positional embeddings | BERT, GPT-2 |
| Rotary Position Embeddings (RoPE) | LLaMA, Mistral |
| ALiBi | MPT, BLOOM |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def positional_encoding(max_len, d_model):
    PE = np.zeros((max_len, d_model))
    positions = np.arange(max_len)[:, np.newaxis]        # (max_len, 1)
    dims = np.arange(d_model)[np.newaxis, :]             # (1, d_model)

    # even dims: sin, odd dims: cos
    angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    PE[:, 0::2] = np.sin(angles[:, 0::2])
    PE[:, 1::2] = np.cos(angles[:, 1::2])
    return PE

max_len, d_model = 50, 64
PE = positional_encoding(max_len, d_model)

plt.figure(figsize=(12, 5))
plt.pcolormesh(PE, cmap='RdBu')
plt.xlabel('Embedding Dimension')
plt.ylabel('Position in Sequence')
plt.title('Sinusoidal Positional Encoding Heatmap')
plt.colorbar(label='Encoding Value')
plt.tight_layout()
plt.show()

print(f"PE shape: {PE.shape}  →  (seq_len={max_len}, d_model={d_model})")

## 7. Full Attention Flow Example

**Sentence:** *"The cat ate the fish"*

```
Step 1  →  Convert words to token embeddings
Step 2  →  Add positional encodings to each embedding
Step 3  →  Generate Q, K, V vectors for each token
Step 4  →  Each token attends to all other tokens (scaled dot-product attention)
Step 5  →  Combine weighted values into context vectors
Step 6  →  Pass through feed-forward network
```

---

### Visual Flow

```
Input Tokens
    ↓
Token Embeddings  +  Positional Encoding
    ↓
┌─────────────────────────────────┐
│         Multi-Head Attention    │
│  Head 1 │ Head 2 │ ... │ Head h │
└─────────────────────────────────┘
    ↓  Concat + Linear
Add & Norm
    ↓
Feed-Forward Network
    ↓
Add & Norm
    ↓
Output Representations
```

## 8. Why This Matters for LLMs

### Long Context Understanding
Attention connects any two positions in the sequence directly — beginning of prompt to end of prompt.

### Parallel Training
Unlike RNNs — all tokens are processed **simultaneously**. Huge training speedup.

### Better Language Understanding
Captures:
- Long-range dependencies
- Grammar
- Semantics
- Coreference

## 9. Common Interview-Level Insights

### Why is Attention better than RNN?

| Aspect | RNN | Attention |
|--------|-----|-----------|
| Training | Sequential (slow) | Parallel (fast) |
| Long dependencies | Degrades with distance | Direct connection |
| Gradient flow | Vanishing gradient problem | Better gradient flow |

---

### Why Multi-Head instead of a bigger single head?

> Different heads learn **different types of relationships** simultaneously.
> A single larger head would try to learn everything in one pass and miss nuanced patterns.

---

### Why is Positional Encoding needed?

> Self-attention is **permutation invariant** — it treats tokens as an unordered set.
> Without positional encoding, shuffling the input produces the same output.
> Position encoding restores **word order awareness**.

## 10. Study Notes — Quick Revision

| Concept | What it does | Key detail |
|---------|-------------|------------|
| **Attention** | Computes relevance between tokens | Uses Q/K/V; weighted sum of Values |
| **Self-Attention** | Attention within the same sequence | Helps with coreference & long-range links |
| **Multi-Head Attention** | Multiple attention mechanisms in parallel | Each head learns diverse relationships |
| **Positional Encoding** | Adds token order information | Needed because transformers lack sequence awareness |

---

### Core Formulas

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,W^O$$

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

## 11. Mental Model Summary

> Think of a Transformer like a **meeting room** where every word is present.

---

**Every word enters the meeting room.**

Each word asks: *"Who here is important for me?"*

| Component | Role in the meeting |
|-----------|-------------------|
| **Attention** | Finds which other words are relevant |
| **Multi-Head Attention** | Multiple specialists analyzing different relationships in parallel |
| **Positional Encoding** | Gives each word its seat number / order |

---

```
"I went to the bank to deposit money."
 ↑                    ↑       ↑
 position 1        position 6  position 7

bank's attention weights:
  deposit → 0.65  ← most relevant
  money   → 0.30
  went    → 0.05

Result: "bank" understood as financial institution ✓
```